# h5ad column extractor

Uses `h5ad_extractor` (backed read) to pull `obs` or `var` columns into Parquet or CSV with the AnnData row index preserved.

In [ ]:
import pandas as pd
import scanpy as sc
from h5ad_extractor import H5adExtractConfig, extract_annotation_columns
from shared.repo import REPO_ROOT

ROOT = REPO_ROOT

## Paths and input file

`ROOT` is [`REPO_ROOT`](../../scripts/shared/repo.py) (repo root from `.git` / `.venv` walk). Adjust `H5AD_PATH` if needed. By default we use `tmp/` when present, otherwise the first raw `.h5ad` under the scBaseCount data directory, otherwise a clustered file under `output/cytetype/data/` if present.

In [ ]:
H5AD_DIR = ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens"

H5AD_PATH = ROOT / "output/cytetype/data/SRX17412841_cytetype_annotated.h5ad"

print(H5AD_PATH)

## Preview available columns

In [ ]:
adata = sc.read(str(H5AD_PATH), backed="r")
try:
    print("obs columns:", list(adata.obs.columns))
    print("n cells", adata.n_obs, " n genes", adata.n_vars)
finally:
    if getattr(adata, "isbacked", False) and adata.file is not None:
        adata.file.close()

## Extract and save

Set `COLUMN_NAMES` to columns that exist in `adata.obs` (or set `annotationAxis="var"` and use `adata.var` names). By default, files go to `output/h5ad_extract/` under the repo root (`H5adExtractConfig.outputDir`), as `{h5ad_stem}_{obs|var}_columns.parquet` or `.csv`. Optional `outputPath` overrides that with a specific file path (still resolved with `REPO_ROOT` when relative). `gs://` h5ad paths are supported.

In [ ]:
COLUMN_NAMES = ["cell_type", "leiden_merged", "cytetype_annotation_leiden_merged"]

pq_path = extract_annotation_columns(
    H5adExtractConfig(
        h5adPath=H5AD_PATH,
        columnNames=COLUMN_NAMES,
        outputFormat="parquet",
    )
)
csv_path = extract_annotation_columns(
    H5adExtractConfig(
        h5adPath=H5AD_PATH,
        columnNames=COLUMN_NAMES,
        outputFormat="csv",
    )
)

print(pq_path)
print(csv_path)

## Load outputs

In [ ]:
df_pq = pd.read_parquet(pq_path)
df_csv = pd.read_csv(csv_path)

df_pq["leiden_merged"] = df_pq["leiden_merged"].astype(int)
df_csv["leiden_merged"] = df_csv["leiden_merged"].astype(int)

# display(df_pq.head())
# print(f"df_pq.shape == df_csv.shape: {df_pq.shape == df_csv.shape}")
# df_pq.shape

# for df in [df_pq, df_csv]:
#     print(df.columns)
#     print(df.shape)
#     print(df.head())

# df_pq["leiden_merged"].astype(int) == df_csv["leiden_merged"].astype(int)

(df_pq == df_csv).value_counts()